In [1]:
import torch
import random
import matplotlib.pyplot as plt
import numpy as np
from env import (SingleLeafThreadEnv, ThreadingConfig)
from loss import TrajectoryBalanceLoss
from gflownet import GFlowNetTrainer, TrajectorySampler, estimate_log_reward_center
from viz import draw_tree_edge_index, draw_local_tree_sequence
from utils import (
    get_max_actions,
    load_argweaver_sites_graph_inputs,
    graph_segments_to_physical_tskit,
    write_graph_segments_json,
)
from models import Policy


In [2]:
SITES_PATH = "./validation_scripts/scripts/argweaver/argweaver_input/sim_l1mb_jc690.sites"
TREES_PATH = "./validation_scripts/trees/sim_l1mb_jc690.trees"
TIME_GRID_SIZE = 50

GENO, REFERENCE_GRAPH_SEGMENTS, LEAF_NAMES, ALL_LEAF_IDS, TIME_GRID = load_argweaver_sites_graph_inputs(
    SITES_PATH,
    TREES_PATH,
    time_grid_size=TIME_GRID_SIZE,
)


In [3]:
TIME_GRID

(0.0,
 4525.717212171848,
 9051.434424343695,
 13577.151636515544,
 18102.86884868739,
 22628.586060859237,
 27154.303273031088,
 31680.020485202935,
 36205.73769737478,
 40731.45490954663,
 45257.172121718475,
 49782.88933389032,
 54308.606546062176,
 58834.32375823402,
 63360.04097040587,
 67885.75818257772,
 72411.47539474956,
 76937.19260692141,
 81462.90981909326,
 85988.6270312651,
 90514.34424343695,
 95040.0614556088,
 99565.77866778064,
 104091.49587995249,
 108617.21309212435,
 113142.9303042962,
 117668.64751646805,
 122194.36472863989,
 126720.08194081174,
 131245.79915298359,
 135771.51636515543,
 140297.23357732728,
 144822.95078949913,
 149348.66800167097,
 153874.38521384282,
 158400.10242601467,
 162925.8196381865,
 167451.53685035836,
 171977.2540625302,
 176502.97127470205,
 181028.6884868739,
 185554.40569904575,
 190080.1229112176,
 194605.84012338944,
 199131.5573355613,
 203657.27454773313,
 208182.99175990498,
 212708.70897207683,
 217234.4261842487,
 221760.143

In [4]:
ALL_LEAF_IDS[-1]

7

In [5]:
## Env

PRINT_ACTIONS = False
SHOW_LOCAL_TREES = False
env_cfg = ThreadingConfig.from_argweaver_sites(
    SITES_PATH,
    TIME_GRID,
    mutation_rate=2e-8,
    recomb_rate=2e-8,
    reward_temperature=1.0,
    substitution_model="JC",
)
env = SingleLeafThreadEnv(env_cfg, ALL_LEAF_IDS, REFERENCE_GRAPH_SEGMENTS)


In [6]:
## Policy
policy_model = Policy(LEAF_NAMES, time_grid=TIME_GRID, device='cuda')

In [ ]:
## N inner episodes with centered Trajectory Balance
## One episode = one focal leaf threaded across all sites.

if not hasattr(policy_model, "log_Z"):
    raise RuntimeError("Policy is missing log_Z; rerun the Policy cell after reloading models.py.")

BATCH_SIZE = 8
N_UPDATES = 100
CALIBRATION_TRAJECTORIES = max(16, 2 * len(ALL_LEAF_IDS))
FOCAL_LEAF_ORDER = [ALL_LEAF_IDS[i % len(ALL_LEAF_IDS)] for i in range(N_UPDATES)]

policy_model.train()

optimizer = torch.optim.Adam(
    [
        {"params": [param for name, param in policy_model.named_parameters() if name != "log_Z"], "lr": 1e-3},
        {"params": [policy_model.log_Z], "lr": 1e-2},
    ]
)

sampler = TrajectorySampler(env, policy_model, reset_to_reference=True)
log_reward_center = estimate_log_reward_center(
    sampler,
    num_trajectories=CALIBRATION_TRAJECTORIES,
    focal_leaves=ALL_LEAF_IDS,
    reset_to_reference=True,
)
tb_loss = TrajectoryBalanceLoss(log_reward_center=log_reward_center)
trainer = GFlowNetTrainer(
    env,
    policy_model,
    optimizer,
    loss_fn=tb_loss,
    sampler=sampler,
    reset_to_reference=True,
    max_grad_norm=10.0,
)

episode_summaries = []
print(f"calibrated log_reward_center: {log_reward_center:.4f}")

for update_idx in range(N_UPDATES):
    focal_leaf = FOCAL_LEAF_ORDER[update_idx]
    metrics = trainer.train_step(
        batch_size=BATCH_SIZE,
        focal_leaf=focal_leaf,
        temperature=1.0,
        reset_to_reference=True,
    )
    trajectories = metrics["trajectories"]
    first_traj = trajectories[0]
    summary = {
        "update": update_idx + 1,
        "focal_leaf": int(focal_leaf),
        "focal_leaf_name": LEAF_NAMES[focal_leaf],
        "steps": int(first_traj.length),
        "loss": float(metrics["loss"]),
        "rms_tb_error": float(metrics["rms_tb_error"]),
        "mean_tb_error": float(metrics["mean_tb_error"]),
        "mean_sum_log_P_F": float(metrics["mean_sum_log_pf"]),
        "mean_sum_log_P_B": float(metrics["mean_sum_log_pb"]),
        "mean_log_reward": float(metrics["mean_log_reward"]),
        "mean_centered_log_reward": float(metrics["mean_centered_log_reward"]),
        "mean_recomb_count": float(metrics["mean_recomb_count"]),
        "log_Z": float(metrics["log_Z"]),
        "log_reward_center": float(metrics["log_reward_center"]),
        "graph_segments": first_traj.terminal_state.current_graph_segments,
    }
    if "grad_norm" in metrics:
        summary["grad_norm"] = float(metrics["grad_norm"])
    episode_summaries.append(summary)

    print(
        f"update {update_idx + 1:03d}/{N_UPDATES} | "
        f"leaf {LEAF_NAMES[focal_leaf]} | batch {BATCH_SIZE} | "
        f"mean_log_R {metrics['mean_log_reward']:.4f} | "
        f"mean_log_P_F {metrics['mean_sum_log_pf']:.4f} | "
        f"mean_log_P_B {metrics['mean_sum_log_pb']:.4f} | "
        f"rms_tb {metrics['rms_tb_error']:.4f} | "
        f"log_Z {metrics['log_Z']:.4f} | loss {metrics['loss']:.4f}"
    )

print("training summary")
print(f"updates: {len(episode_summaries)}")
print("focal leaf order: " + ", ".join(f"{LEAF_NAMES[lid]} ({lid})" for lid in FOCAL_LEAF_ORDER))
print(f"final log_Z: {policy_model.log_Z.item():.4f}")
print(f"log_reward_center: {tb_loss.log_reward_center:.4f}")
print(f"final graph segments: {len(env.current_graph_segments)}")


calibrated log_reward_center: -141465.3379
update 001/100 | leaf spl0_1 | batch 8 | mean_log_R -88139.2891 | mean_log_P_F -9832.5469 | mean_log_P_B -63163.3438 | rms_tb 580.5312 | log_Z -0.0100 | loss 337016.2500
update 002/100 | leaf spl0_2 | batch 8 | mean_log_R -88225.1797 | mean_log_P_F -9846.2744 | mean_log_P_B -63191.4180 | rms_tb 432.8789 | log_Z -0.0175 | loss 187385.7969
update 003/100 | leaf spl1_1 | batch 8 | mean_log_R -88002.6328 | mean_log_P_F -9834.6201 | mean_log_P_B -63027.4414 | rms_tb 680.5713 | log_Z -0.0197 | loss 463175.7500
update 004/100 | leaf spl1_2 | batch 8 | mean_log_R -88187.4688 | mean_log_P_F -9825.4746 | mean_log_P_B -63092.4648 | rms_tb 379.4694 | log_Z -0.0210 | loss 143996.7656
update 005/100 | leaf spl2_1 | batch 8 | mean_log_R -88043.2266 | mean_log_P_F -9842.7285 | mean_log_P_B -62950.0703 | rms_tb 561.7978 | log_Z -0.0167 | loss 315619.2500
update 006/100 | leaf spl2_2 | batch 8 | mean_log_R -88050.8359 | mean_log_P_F -9823.5889 | mean_log_P_B -6

In [12]:
## Save trained policy
from pathlib import Path

CHECKPOINT_DIR = Path("./checkpoints")
CHECKPOINT_DIR.mkdir(exist_ok=True)
CHECKPOINT_PATH = CHECKPOINT_DIR / "policy.pt"

torch.save(
    {
        "state_dict": policy_model.state_dict(),
        "leaf_names": LEAF_NAMES,
        "time_grid": list(TIME_GRID),
        "log_Z": float(policy_model.log_Z.item()),
        "log_reward_center": float(tb_loss.log_reward_center),
        "n_updates_trained": N_UPDATES,
    },
    CHECKPOINT_PATH,
)
print(f"Saved policy to {CHECKPOINT_PATH}")

In [13]:
## Load checkpoint, sample diverse trajectories, and save physical-coordinate .trees outputs
from pathlib import Path
import json
from gflownet import TrajectorySampler
from utils import graph_segments_to_physical_tskit, write_graph_segments_json

loaded_policy = Policy(LEAF_NAMES, time_grid=TIME_GRID, device='cuda')
payload = torch.load(CHECKPOINT_PATH, map_location='cuda')
state_dict = payload.get("state_dict", payload) if isinstance(payload, dict) else payload
loaded_policy.load_state_dict(state_dict)
loaded_policy.eval()

sampler = TrajectorySampler(env, loaded_policy, reset_to_reference=True)

N_SAMPLES = 100
focal_leaf_for_demo = ALL_LEAF_IDS[0]
OUTPUT_DIR = Path("./validation_scripts/scripts/drag/l25kb")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
physical_sequence_length = (
    float(env.config.genomic_region[1])
    if env.config.genomic_region is not None
    else float(env.config.sequence_length)
)

sampled_trajectories = []
trace_rows = []

print(f"Sampling {N_SAMPLES} trajectories for focal leaf {LEAF_NAMES[focal_leaf_for_demo]}:")
for i in range(N_SAMPLES):
    traj = sampler.sample(
        focal_leaf=focal_leaf_for_demo,
        temperature=1.0,
        greedy=False,
        reset_to_reference=True,
    )
    sampled_trajectories.append(traj)

    trees_path = OUTPUT_DIR / f"sample_{i:03d}.trees"
    segments_path = OUTPUT_DIR / f"sample_{i:03d}.segments.json"
    segments = traj.terminal_state.current_graph_segments
    graph_segments_to_physical_tskit(
        segments,
        num_samples=len(ALL_LEAF_IDS),
        sequence_length=physical_sequence_length,
        output_path=trees_path,
    )
    write_graph_segments_json(
        segments_path,
        segments,
        sequence_length=physical_sequence_length,
        num_samples=len(ALL_LEAF_IDS),
        coordinate_mode="physical",
    )

    trace_rows.append(
        {
            "sample_index": i,
            "focal_leaf": int(focal_leaf_for_demo),
            "focal_leaf_name": LEAF_NAMES[focal_leaf_for_demo],
            "log_reward": float(traj.log_reward.detach().cpu().item()),
            "sum_log_pf": float(traj.sum_log_pf.detach().cpu().item()),
            "recomb_count": int(traj.recomb_count),
            "length": int(traj.length),
            "checkpoint_path": str(CHECKPOINT_PATH),
            "sites_path": str(SITES_PATH),
            "trees_path": str(TREES_PATH),
            "sample_trees_path": str(trees_path),
            "segments_json_path": str(segments_path),
        }
    )

    actions_preview = traj.actions[:8]
    suffix = "..." if traj.length > 8 else ""
    print(
        f"  sample {i+1}: log_reward={traj.log_reward.item():.4f} | "
        f"sum_log_pf={traj.sum_log_pf.item():.4f} | "
        f"recombs={traj.recomb_count} | length={traj.length} | "
        f"saved={trees_path} | actions={actions_preview}{suffix}"
    )

trace_path = OUTPUT_DIR / "ours_trace.json"
with trace_path.open("w", encoding="utf-8") as handle:
    json.dump(trace_rows, handle, indent=2)
print(f"Saved {len(sampled_trajectories)} sampled .trees files and trace to {OUTPUT_DIR}")
